In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
# 메모리 기능
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

In [8]:


# LLM 생성
llm=ChatOpenAI(model="gpt-4o-mini") 

store={}    # 빈 dictionary 생성, session_id 저장, session_id:LLM 입장에서는 사용자 구분을 의미
# {session_id:InMemoryChatMessageHistory(),session_id:InMemoryChatMessageHistory(), ...}

# 사용자별(session_id) 대화(메시지) 저장소를 반환한다.
def get_session_history(session_id:str)->BaseChatMessageHistory:    # chatmessagehistory의 기본 기능. 보통 base 있으면 자식 클래스가 있음.
                        # 매개변수 타입 힌트     # return 타입의 타입 힌트
    if session_id not in store:     # 없으면 새로 들어온 사용자(session)
        # 사용자별 MessageHistory를 주겠다.
        store[session_id]=InMemoryChatMessageHistory()    # 사용자 별로 저장소(InMemoryChatMessageHistory) 만들겠다.
        # 키:사용자, 값:메시지 이력
    return store[session_id]    # store 딕셔너리

# 저장소를 사용(실행:run)하는 게 필요.
with_message_history = RunnableWithMessageHistory(llm, get_session_history)     # 알아서 append할 것.

# 사용자 session_id 설정: config 설정
config = {"configurable":{"session_id":"abc2"}} # session_id='abc2'

response = with_message_history.invoke(
    [HumanMessage(content="안녕. 난 윤호야.")],
    config=config  # 공간(방) 생성.
)

print(response.content)

안녕하세요, 윤호님! 어떻게 도와드릴까요?


In [9]:
response=with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config
)

In [12]:
print(response.content)

당신의 이름은 윤호입니다. 제가 도와드릴 수 있는 것이 있나요?


In [19]:
# 세션 변경 진행
id="abc2"
config={"configurable":{"session_id":f"{id}"}}

In [20]:
response=with_message_history.invoke(
    [HumanMessage(content="내 이름 뭐라고?")],
    config=config
)
print(response.content)

당신의 이름은 윤호입니다. 더 궁금한 점이 있으신가요?


In [22]:
# session_id='abc2'
response = with_message_history.invoke(
    [HumanMessage(content="내가 어느 나라 사람인지 맞혀보고, 그 나라의 문화에 대해 말해봐")],
    config=config       # session_id="abc2"
)
print(response.content)

윤호라는 이름은 주로 한국에서 사용되므로, 한국 사람일 가능성이 높습니다. 

한국 문화는 풍부하고 다양합니다. 여기 몇 가지 주요 특징을 소개할게요:

1. **음식**: 한국 음식은 양념이 풍부하고 다양합니다. 김치, 비빔밥, 불고기 등이 유명하며, 식사는 보통 밥, 국, 여러 가지 반찬으로 구성됩니다.

2. **전통**: 한국의 전통 문화는 유교의 영향을 많이 받았습니다. 세배(새해 인사), 설날과 추석 같은 명절이 중요하며, 한복과 같은 전통 의상도 여전히 특별한 날에 착용됩니다.

3. **K-POP과 대중문화**: 최근 몇 년간 K-POP, 드라마, 영화 등이 세계적으로 인기를 끌며 한국 문화의 대중화에 기여했습니다. 방탄소년단(BTS)이나 블랙핑크와 같은 그룹이 예시입니다.

4. **가족 중심**: 한국 사회는 가족 중심적인 경향이 강하여, 가족 모임과 공동체 활동이 중요시됩니다.

이 외에도 한국에는 많은 흥미로운 문화유산과 전통이 있습니다. 더 궁금한 점이 있으신가요?


In [23]:
for resp in with_message_history.stream(
    [HumanMessage(content="내가 어느 나라 사람인지 맞혀보고, 그 나라의 문화에 대해 말해봐. 참고로 나는 한국 사람은 아니야.")],
    config=config
):
    print(resp.content, end="|")

|알|겠습니다|!| 그렇|다면| 윤|호|라는| 이름|이| 한국|이| 아닌| 다른| 나라|에서| 사용|될| 수도| 있|겠|군|요|.| 예|를| 들어|,| 일본|,| 중국|,| 또는| 다른| 아|시아| 국가|에서| 사용|될| 수| 있습니다|.| 그러나| 정확|한| 나라|를| 맞|히|기는| 어렵|네요|.

|만|약| 일본| 사람|이라고| 가|정|해|보|면|,| 일본| 문화|에| 대해| 설명|해|드|리|겠습니다|.

|###| 일본| 문화|의| 특징|

|1|.| **|전|통|과| 현대|의| 조|화|**|:| 일본|은| 전|통|적인| 문화|와| 현대|적인| 문화|가| 잘| 어|우|러|져| 있습니다|.| 사|무|라이| 문화|와| 다|도| 같은| 전|통|이| 여|전히| 존|중|받|는| 한|편|,| 애|니|메|이션|과| 영화| 산업|도| 세계|적으로| 유명|합니다|.

|2|.| **|음|식|**|:| 일본|의| 음식|은| 신|선|함|과| 조|화를| 중요|시|합니다|.| 스|시|,| 사|시|미|,| 라|멘|,| 텐|푸|라| 등이| 널|리| 알려|져| 있으며|,| 계|절| 재|료|를| 활용|한| 요|리가| 많|습니다|.

|3|.| **|예|의|와| 존|중|**|:| 일본| 사회|에서는| 예|의|와| 존|중|이| 중요한| 가치|입니다|.| 인|사|할| 때|는| 고|개|를| 숙|이며|,| 대|화| 중|에는| 상대|방|을| 배|려|하는| 것이| 일반|적|입니다|.

|4|.| **|축|제|와| 전|통| 행사|**|:| 일본|에는| 다양한| 전|통| 축|제|(|마|츠|리|)가| 있으며|,| 벚|꽃|놀|이나| 여|름| 축|제| 때|는| 많은| 사람들이| 모|여| 즐|깁|니다|.

|어|떤| 나라| 출|신|인지| 알려|주|시면| 그| 나라|에| 대해| 더| 구|체|적으로| 이야기|해|드|릴| 수| 있어|요|!||||